# 04 TypeB 上機教材：你會批准這個 AI 交易策略上線嗎？

**Big HOT：如果你是投資委員會成員，你會批准這個 AI 交易策略使用真實資金嗎？**

最後一堂課不是再追求更漂亮的模型分數，而是把策略放到風險、解釋性、治理與溝通的框架中審查。


## 課前閱讀與 LMS 回應

課前請先閱讀 [Week 8 main](../../08/week8_main.md)。

課前 HOT：

```text
If a strategy beats the benchmark in one backtest but fails when transaction cost doubles,
would you approve it?
```

LMS 回應格式：

```text
My initial decision:
approve small pilot / delay / reject

My reason:
The biggest risk is ______.

The evidence that would change my decision is ______.
```

課中使用方式：開場做 initial vote，課末做 final vote，比較 risk table、stress matrix 與 feature explanation 如何改變決策。


## 3 小時 HOT storyline

| 時間 | Cluster | 核心 HOT | 可見產出 |
|---:|---|---|---|
| 0:00-0:25 | C1. Investment committee frame | 你會批准、暫緩、還是拒絕？ | initial vote |
| 0:25-1:05 | C1. Risk diagnosis | 哪個風險指標最可能改變決策？ | risk table |
| 1:05-1:45 | C2. Stress and regime | 成本、threshold、子期間改變後策略還站得住嗎？ | stress matrix |
| 1:45-2:20 | C2. Explainability | 模型依賴的特徵合理嗎？ | feature explanation |
| 2:20-2:50 | C3. Strategy proposal | 如何誠實寫一頁策略提案？ | one-page proposal |
| 2:50-3:00 | C4. Final vote | 什麼 evidence 讓你改變投票？ | final decision |


## Learning Loop Map（新版 Type B 操作版）

| Loop | Mini-input | BIT / HOT | Visible output | Delayed feedback focus |
|---|---|---|---|---|
| 1 | Investment committee frame | Vote + justify | initial vote | approve / delay / reject 需要 evidence |
| 2 | Risk diagnosis | Rank/order | risk metric ranking | 不同角色關心不同風險 |
| 3 | Stress and regime | Stress / compare | stress matrix 與 subperiod statement | robust pattern vs cherry-picking |
| 4 | Explainability | Explain + challenge | feature explanation note | feature importance 不是因果 |
| 5 | Proposal language | Critique + revise | risk-aware revision | 避免過度宣稱 |
| 6 | Final decision | Synthesize / defend / reflect | one-page proposal + final vote | evidence 如何改變決策 |


## TypeB 課堂語言

```text
I approve / delay / reject because ______.
The biggest hidden risk is ______.
This result is fragile because ______.
The model explanation is credible / not credible because ______.
Before live trading, I would require ______.
```


## 學生回應方式（課中使用）

本堂課每個回答都要像投資委員會：做決策，但也揭露限制。

| 場景 | 回應格式 |
|---|---|
| Initial vote | `I approve / delay / reject because ______.` |
| Risk ranking | `The most important risk is ______ because ______.` |
| XAI | `This feature importance is useful, but it does not prove ______.` |
| Final vote | `My decision changed / did not change because ______.` |

同儕質詢只能問 evidence-based questions，例如：

```text
Which table supports that claim?
What happens if transaction cost doubles?
What would make you reject the strategy?
```


In [ ]:
# 課堂穩定性設定：預設使用合成 OHLCV 資料。
# 若教室網路穩定且已安裝 yfinance，可以把 USE_ONLINE_DATA 改成 True。
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

SYMBOL = "0050.TW"
USE_ONLINE_DATA = False


def make_synthetic_ohlcv(n=760, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2021-01-01", periods=n)
    t = np.arange(n)
    regime = np.select(
        [t < n * 0.33, t < n * 0.66],
        [0.00045, -0.00015],
        default=0.00025,
    )
    shocks = rng.normal(0, 0.011, n)
    shocks[0] = rng.normal(0, 0.011)
    ret = regime + shocks + 0.08 * np.r_[0, shocks[:-1]]
    close = 100 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.012, n))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.012, n))
    volume = rng.lognormal(mean=15.2, sigma=0.25, size=n) * (1 + 8 * np.abs(ret))
    df = pd.DataFrame(
        {
            "Open": open_,
            "High": high,
            "Low": low,
            "Close": close,
            "Adj Close": close,
            "Volume": volume.astype(int),
        },
        index=dates,
    )
    df.index.name = "Date"
    return df


def load_market_data(symbol=SYMBOL, start="2020-01-01", use_online=USE_ONLINE_DATA):
    if use_online:
        try:
            import yfinance as yf
            df = yf.download(symbol, start=start, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if not df.empty and {"Open", "High", "Low", "Close", "Volume"}.issubset(df.columns):
                print(f"Loaded online data: {symbol}, rows={len(df)}")
                return df.dropna()
        except Exception as exc:
            print("Online download failed; falling back to synthetic data.")
            print(type(exc).__name__, exc)
    print("Using synthetic OHLCV data. Toggle USE_ONLINE_DATA=True for real market data.")
    return make_synthetic_ohlcv()


def add_features(raw):
    df = raw.copy()
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_fwd_1d"] = df["Close"].shift(-1) / df["Close"] - 1
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_gap"] = df["ma_5"] / df["ma_20"] - 1
    df["mom_5"] = df["Close"] / df["Close"].shift(5) - 1
    df["mom_20"] = df["Close"] / df["Close"].shift(20) - 1
    df["vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    volume_mean = df["Volume"].rolling(20).mean()
    volume_std = df["Volume"].rolling(20).std()
    df["volume_z"] = (df["Volume"] - volume_mean) / volume_std
    df = df.dropna()
    df["target_up"] = (df["ret_fwd_1d"] > 0).astype(int)
    return df


def max_drawdown(ret):
    wealth = (1 + ret.fillna(0)).cumprod()
    dd = wealth / wealth.cummax() - 1
    return dd.min()


def sharpe(ret, periods=252):
    vol = ret.std()
    if vol == 0 or np.isnan(vol):
        return np.nan
    return np.sqrt(periods) * ret.mean() / vol


def perf_table(returns_dict):
    rows = []
    for name, ret in returns_dict.items():
        ret = pd.Series(ret).dropna()
        rows.append(
            {
                "strategy": name,
                "ann_return": (1 + ret).prod() ** (252 / len(ret)) - 1 if len(ret) else np.nan,
                "ann_vol": ret.std() * np.sqrt(252),
                "sharpe": sharpe(ret),
                "max_drawdown": max_drawdown(ret),
                "win_rate": (ret > 0).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("strategy").round(4)


raw = load_market_data()
df = add_features(raw)
df.tail()


## 建立候選策略

為了讓 Week 8 可獨立執行，這裡建立一個簡單 AI 候選策略，後續用風險審查它。


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

feature_cols = ["ma_gap", "mom_5", "mom_20", "vol_20", "volume_z", "range_pct"]
split_idx = int(len(df) * 0.70)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()

model = RandomForestClassifier(n_estimators=250, max_depth=4, random_state=21)
model.fit(train[feature_cols], train["target_up"])
test["prob_up"] = model.predict_proba(test[feature_cols])[:, 1]

def make_position(prob_up, threshold=0.55):
    return pd.Series(np.where(prob_up > threshold, 1.0, 0.0), index=prob_up.index)

def strategy_returns(market, threshold=0.55, cost=0.001):
    pos = make_position(market["prob_up"], threshold=threshold)
    turnover = pos.diff().abs().fillna(pos.abs())
    net = pos * market["ret_fwd_1d"] - turnover * cost
    return pd.DataFrame({"position": pos, "turnover": turnover, "net_ret": net})

candidate = strategy_returns(test, threshold=0.55, cost=0.001)
perf_table({"candidate": candidate["net_ret"], "buy_and_hold": test["ret_fwd_1d"]})


## C1 HOT 1：你現在會批准、暫緩、還是拒絕？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Vote + justify |
| Think | 60 秒：個人投 approve / delay / reject。 |
| Pair/Group | 3 分鐘：小組比較需要什麼 evidence 才能改票。 |
| Visible output | initial vote + reason。 |
| Delayed feedback | 先不判對錯，追問：What evidence would change your vote? |

HOT 類型：**Vote + Justify**  
學生先用有限資訊投票。Week 8 最後再投一次，觀察 evidence 如何改變判斷。


In [ ]:
initial_vote = pd.DataFrame(
    {
        "decision": ["approve", "delay", "reject"],
        "what_evidence_would_support": [
            "stable after costs and explainable",
            "more subperiod and stress tests",
            "large drawdown, leakage, or no benchmark edge",
        ],
        "student_count": [0, 0, 0],
    }
)
initial_vote


## C1 HOT 2：哪個風險指標最可能改變你的決策？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Rank/order |
| Think | 60 秒：個人排序最重要風險指標。 |
| Pair/Group | 4 分鐘：小組以投委會角度辯護排序。 |
| Visible output | risk metric ranking。 |
| Delayed feedback | 比較研究員與風控角色的不同優先順序。 |

HOT 類型：**Rank + Defend**  
不同角色會看不同風險：研究員看 Sharpe，投委會可能先看 drawdown、turnover、可解釋性。


In [ ]:
risk_metrics = perf_table({"candidate": candidate["net_ret"], "buy_and_hold": test["ret_fwd_1d"]})
risk_metrics["avg_turnover"] = [candidate["turnover"].mean(), 0.0]
risk_metrics["active_days"] = [(candidate["position"] != 0).mean(), 1.0]
risk_metrics.round(4)


In [ ]:
equity = pd.DataFrame(
    {
        "candidate": (1 + candidate["net_ret"].fillna(0)).cumprod(),
        "buy_and_hold": (1 + test["ret_fwd_1d"].fillna(0)).cumprod(),
    }
)
drawdown = equity / equity.cummax() - 1

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
equity.plot(ax=axes[0], title="Equity curve")
drawdown.plot(ax=axes[1], title="Drawdown")
plt.tight_layout()
plt.show()


## C2 HOT 3：成本與 threshold 一改，策略是否還成立？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Stress test + interpret |
| Think | 60 秒：預測成本與 threshold 改變後結論是否穩定。 |
| Pair/Group | 4 分鐘：小組解讀 stress matrix。 |
| Visible output | stress matrix interpretation。 |
| Delayed feedback | 追問：Are you reading the pattern or cherry-picking one cell? |

HOT 類型：**Stress test + Interpret**  
一個策略若只在單一成本、單一 threshold 好看，投委會通常會要求暫緩。


In [ ]:
stress_rows = []
for threshold in [0.50, 0.55, 0.60, 0.65]:
    for cost in [0.0, 0.001, 0.002, 0.003]:
        bt = strategy_returns(test, threshold=threshold, cost=cost)
        stress_rows.append(
            {
                "threshold": threshold,
                "cost": cost,
                "sharpe": sharpe(bt["net_ret"]),
                "max_drawdown": max_drawdown(bt["net_ret"]),
                "ann_return": (1 + bt["net_ret"]).prod() ** (252 / len(bt)) - 1,
                "avg_turnover": bt["turnover"].mean(),
            }
        )
stress = pd.DataFrame(stress_rows)
stress.pivot(index="threshold", columns="cost", values="sharpe").round(3)


## C2 HOT 4：策略是否只在某一段市場有效？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare + challenge |
| Think | 60 秒：個人判斷是否有 regime risk。 |
| Pair/Group | 4 分鐘：小組比較子期間結果。 |
| Visible output | subperiod risk statement。 |
| Delayed feedback | 問：Do we reject, delay, or ask for more data? |

HOT 類型：**Compare + Challenge**  
學生要判斷：子期間結果不穩，是策略失效，還是樣本太短？


In [ ]:
cut_points = np.linspace(0, len(candidate), 4, dtype=int)
regime_rows = []
for i in range(3):
    part = candidate.iloc[cut_points[i] : cut_points[i + 1]]
    bh_part = test.iloc[cut_points[i] : cut_points[i + 1]]["ret_fwd_1d"]
    regime_rows.append(
        {
            "subperiod": i + 1,
            "start": part.index.min().date(),
            "end": part.index.max().date(),
            "candidate_sharpe": sharpe(part["net_ret"]),
            "candidate_mdd": max_drawdown(part["net_ret"]),
            "buy_hold_sharpe": sharpe(bh_part),
        }
    )
pd.DataFrame(regime_rows).round(4)


## C2 HOT 5：模型依賴的特徵，金融上說得通嗎？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Explain + challenge |
| Think | 60 秒：先指出最合理與最可疑的重要特徵。 |
| Pair/Group | 4 分鐘：小組連結 feature importance 與金融直覺。 |
| Visible output | feature explanation note。 |
| Delayed feedback | 提醒：importance is not causality。 |

HOT 類型：**Explain + Challenge**  
解釋性不是裝飾；如果模型主要依賴難以事前取得或金融直覺薄弱的特徵，部署風險會上升。


In [ ]:
perm = permutation_importance(
    model,
    test[feature_cols],
    test["target_up"],
    n_repeats=20,
    random_state=21,
    scoring="balanced_accuracy",
)
importance = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }
).sort_values("importance_mean", ascending=False)
importance.round(4)


## C2 HOT 6：AI 文字說明哪裡過度自信？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Critique + revise |
| Think | 60 秒：標出過度自信語句。 |
| Pair/Group | 4 分鐘：小組改寫成風險揭露語句。 |
| Visible output | risk-aware revision。 |
| Delayed feedback | 比較兩種改寫，問哪個更符合 evidence。 |

HOT 類型：**Critique + Revise**  
請學生把過度保證的語句改成風險揭露語句。


In [ ]:
overconfident_language = pd.DataFrame(
    [
        ["The model proves momentum works.", "The model provides limited evidence that momentum-related features may help in this sample."],
        ["The strategy can generate stable profit.", "The strategy requires further stress testing before any live deployment."],
        ["The AI found the best threshold.", "The threshold is a design choice and may be sensitive to costs and sample period."],
        ["Feature importance explains causality.", "Feature importance suggests association and should be checked with domain reasoning."],
    ],
    columns=["overconfident_sentence", "risk_aware_revision"],
)
overconfident_language


## C3 HOT 7：一頁策略提案應該誠實揭露什麼？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Synthesize |
| Think | 90 秒：個人先填 strategy hypothesis 與 biggest risk。 |
| Pair/Group | 6 分鐘：小組完成一頁提案草稿。 |
| Visible output | one-page proposal draft。 |
| Delayed feedback | 教師巡視時找出 overclaim 與 missing evidence。 |

HOT 類型：**Synthesize**  
最終產出不是「模型準確率」，而是一頁能被審查的策略提案。


In [ ]:
proposal = pd.DataFrame(
    {
        "section": [
            "Strategy hypothesis",
            "Data and features",
            "Signal and position rule",
            "Backtest period and benchmark",
            "Risk metrics",
            "Stress test result",
            "Explainability",
            "Deployment decision",
            "Required next evidence",
        ],
        "draft_content": [
            "Momentum and risk-state features may help time long exposure.",
            ", ".join(feature_cols),
            "Long only when prob_up > 0.55; otherwise cash.",
            f"Test period {test.index.min().date()} to {test.index.max().date()}, benchmark buy-and-hold.",
            "Use annual return, Sharpe, max drawdown, turnover, active days.",
            "Check threshold and transaction cost grid.",
            "Use permutation importance and financial intuition check.",
            "Delay / approve / reject based on committee vote.",
            "Longer out-of-sample test and more realistic cost model.",
        ],
    }
)
proposal


## C3 HOT 8：同儕投委會質詢

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Defend + revise |
| Think | 60 秒：每組準備一個 evidence-based question。 |
| Pair/Group | 6 分鐘：同儕投委會質詢。 |
| Visible output | peer review rubric。 |
| Delayed feedback | 先讓被問組回應，再由提問組判斷 evidence 是否充分。 |

HOT 類型：**Defend + Revise**  
每組收到三種問題：一個資料問題、一個風險問題、一個治理問題。回答時必須引用 notebook 中的表或圖。


In [ ]:
peer_review_rubric = pd.DataFrame(
    {
        "criterion": [
            "testable hypothesis",
            "no obvious leakage",
            "benchmark and cost included",
            "risk metrics interpreted honestly",
            "stress tests discussed",
            "model explanation is not overclaimed",
            "deployment decision matches evidence",
        ],
        "score_0_2": "",
        "evidence_cell_or_table": "",
        "revision_needed": "",
    }
)
peer_review_rubric


## C4 HOT 9：最後投票，什麼 evidence 讓你改變立場？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Reflect + decide |
| Think | 60 秒：個人重新投票並說明是否改變。 |
| Pair/Group | 3 分鐘：小組整理 final decision。 |
| Visible output | final vote + next evidence。 |
| Delayed feedback | 最後追問：What evidence changed your decision? |

HOT 類型：**Reflect + Decide**  
學生必須用一句話說明自己投票是否改變，以及改變的原因。


In [ ]:
final_vote = pd.DataFrame(
    {
        "decision": ["approve small pilot", "delay", "reject"],
        "minimum_condition": [
            "only with capital limit, monitoring, and pre-defined stop review",
            "needs more out-of-sample and cost validation",
            "risk or evidence does not justify deployment",
        ],
        "student_count": [0, 0, 0],
    }
)
final_vote


## Optional HOT：3 小時內時間不夠時可跳過

1. **Risk owner**：資料、模型、交易、治理風險各由誰負責？
2. **Monitoring rule**：策略上線後，什麼指標觸發暫停？
3. **Explainability depth**：feature importance 夠不夠，何時需要 SHAP？
4. **Ethics**：使用 GenAI 產生研究假說時，哪些內容必須揭露？


In [ ]:
monitoring_rules = pd.DataFrame(
    [
        ["drawdown", "pause if drawdown worse than research max drawdown by 50%"],
        ["turnover", "review if turnover doubles expected level"],
        ["feature drift", "review if key feature distribution shifts materially"],
        ["cost slippage", "pause if realized cost exceeds assumption for 20 trading days"],
        ["model confidence", "review if probability distribution collapses near 0.5"],
    ],
    columns=["monitor", "example_rule"],
)
monitoring_rules


## Appendix HOT A1：Permutation importance 與 SHAP 的差異是什麼？

HOT 類型：**Compare**  
這裡先用 permutation importance；SHAP 可作為更進階的局部解釋工具。


In [ ]:
xai_compare = pd.DataFrame(
    [
        ["permutation importance", "feature-level performance drop", "simple, model-agnostic", "can be unstable with correlated features"],
        ["SHAP", "local contribution to individual predictions", "detailed explanation", "more complex and easy to over-interpret"],
        ["coefficients", "linear model weight", "transparent", "only fits linear assumptions"],
    ],
    columns=["method", "answers", "strength", "caution"],
)
xai_compare


## Appendix HOT A2：Momentum + sentiment 策略如何接到本課架構？

HOT 類型：**Design**  
把文字情緒視為 alternative data，它仍然要通過 timestamp、feature、cost、benchmark、risk 審查。


In [ ]:
sentiment_strategy_design = pd.DataFrame(
    [
        ["raw text", "news/social/media reports", "publication timestamp must be before trade"],
        ["sentiment score", "positive/negative tone", "model drift and language domain risk"],
        ["momentum interaction", "sentiment confirms trend", "may overfit interaction"],
        ["portfolio policy", "weight changes by signal strength", "turnover and concentration risk"],
    ],
    columns=["component", "role", "main_risk"],
)
sentiment_strategy_design


## Appendix HOT A3：VaR、CVaR、downside risk 會改變決策嗎？

HOT 類型：**Calculate + Interpret**  
平均績效好，不代表尾端風險可以接受。


In [ ]:
def var_cvar(ret, alpha=0.05):
    q = ret.quantile(alpha)
    cvar = ret[ret <= q].mean()
    return q, cvar

tail_rows = []
for name, ret in {"candidate": candidate["net_ret"], "buy_and_hold": test["ret_fwd_1d"]}.items():
    q, cvar = var_cvar(ret.dropna(), alpha=0.05)
    tail_rows.append(
        {
            "strategy": name,
            "daily_VaR_5pct": q,
            "daily_CVaR_5pct": cvar,
            "downside_vol": ret[ret < 0].std() * np.sqrt(252),
        }
    )
pd.DataFrame(tail_rows).set_index("strategy").round(4)


## Appendix HOT A4：治理清單：誰能按下上線按鈕？

HOT 類型：**Governance design**  
AI 交易不是模型人員說可以就可以，還需要風險、合規、投資決策與監控流程。


In [ ]:
governance_checklist = pd.DataFrame(
    [
        ["researcher", "documents hypothesis, data, model, backtest", "research memo"],
        ["risk manager", "reviews drawdown, stress, exposure, model risk", "risk sign-off"],
        ["portfolio manager", "decides capital allocation and benchmark fit", "allocation decision"],
        ["compliance/governance", "checks data use, disclosure, audit trail", "governance approval"],
        ["operations", "monitors execution and slippage", "live monitoring report"],
    ],
    columns=["role", "responsibility", "required_output"],
)
governance_checklist


## Learning Evidence Checklist

本堂課結束前，至少留下這些 evidence：

- [ ] initial approve / delay / reject vote。
- [ ] risk metric ranking。
- [ ] stress matrix interpretation。
- [ ] subperiod risk statement。
- [ ] feature explanation note。
- [ ] risk-aware revision of overconfident language。
- [ ] one-page proposal draft。
- [ ] final vote and next-evidence statement。

Exit ticket：

```text
My final decision is ______ because ______.
The evidence that changed my view is ______.
Before live trading, I would require ______.
```


## 參考資料

本課程設計參考下列概念來源，重點不是要求學生讀完整篇，而是把研究中的核心判斷轉成上機問題。

- López de Prado, M. (2018). *Advances in Financial Machine Learning*. 用於 financial ML failure、labeling、triple-barrier、meta-labeling、finance cross-validation、backtest overfitting。
- Gu, S., Kelly, B., & Xiu, D. (2020). Empirical Asset Pricing via Machine Learning. *Review of Financial Studies*. 用於 momentum、liquidity、volatility、非線性模型與資產報酬預測。
- Machine Learning and Portfolio Optimization 相關文獻。用於 regularization、cross-validation、estimation error 與 portfolio construction。
- Robust perspective on transaction costs in portfolio optimization 相關技術筆記。用於 transaction cost、turnover、robustness。
- XAI in finance 綜述文獻。用於 feature importance、SHAP、trust、risk assessment、governance。
- Lee, W.-Y. momentum-based sentiment trading strategy 相關研究。用於 Appendix 中 momentum + sentiment、benchmark、transaction cost、long-horizon evaluation。
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 用於 practical ML workflow、train/test、validation、classification metrics、error analysis。
